# The *sommerhus*, characterized dynamically

Notebook 3 gave the house a **timeline**. This notebook asks what the timing of its emissions
does to the **impact**, with the dynamic characterization from the previous session - now on a
real inventory instead of a handful of dummy rows.

Two things are yours to work out: how the **time horizon** is counted (section 3), and a
**characterization function of your own** (section 4).

```mermaid
flowchart LR
    tree(🌲 timber tree):::fg-->timber
    timber(🪵 structural timber production):::fg-->construction
    glass_wool(🧵 market for glass wool mat):::ei-->construction
    construction(🏗️ sommerhus construction):::fg-->living
    heat_pump(🔥 heat pump, brine-water 10kW):::ei-->living
    grid(⚡ market group for electricity, low voltage):::ei-->living
    living(🏠 living in the sommerhus):::fg-->waste_wood(🔥 incineration of waste wood):::fg
    living-->fu(FU: 50 years of habitation)

    classDef ei color:#222832, fill:#3fb1c5, stroke:none;
    classDef fg color:#222832, fill:#9c5ffd, stroke:none;
```

<span style="color:#9c5ffd">■</span> foreground &nbsp;&nbsp;
<span style="color:#3fb1c5">■</span> background (premise vintages 2020 / 2030 / 2040 / 2050)



## 0 | The model from notebook 3

System and temporal information are imported rather than retyped -
[`sommerhus_system.py`](../../sommerhus_system.py) and
[`sommerhus_temporal.py`](../../sommerhus_temporal.py) hold exactly what you wrote by hand before.


In [ ]:
import bw2data as bd

from sommerhus_system import build_system
from sommerhus_temporal import add_temporal_information

LIFETIME = 50
BG_DATABASE = "ei_cutoff_3.12_remind-eu_SSP2-NDC_2020"
METHOD = ("IPCC 2021", "climate change", "GWP 100a, incl. H and bio CO2")

build_system(
    lifetime=LIFETIME,
    timber_volume=12,
    insulation_mass=1200,
    heat_pumps=3,
    electricity_per_year=1400,
    background_database=BG_DATABASE,
    method=METHOD,
)
add_temporal_information(lifetime=LIFETIME, background_database=BG_DATABASE)

living = bd.get_node(database="foreground", name="living in the sommerhus")

In [ ]:
from bw_timex import TimexLCA

tlca = TimexLCA({living: 1}, METHOD)
tlca.build_timeline(starting_datetime="2025-01-01", temporal_grouping="month")
tlca.lci()
tlca.static_lcia()

print(f"time-explicit, static characterization: {tlca.static_score:,.0f} kg CO2-eq")

## 1 | The inventory, before any characterization

`tlca.dynamic_inventory_df` is the `date` / `amount` / `flow` / `activity` dataframe you
already know - only this one comes out of a real supply chain. The tree's uptake sits decades
before the house exists. (Positive here means *taken out of the air*: this is a natural
resource flow, and the sign flips once it is characterized.)


In [ ]:
import matplotlib.pyplot as plt

uptake_flow = bd.get_node(
    database=bd.config.biosphere,
    name="Carbon dioxide, in air",
    categories=("natural resource", "in air"),
)
uptake_over_time = (
    tlca.dynamic_inventory_df[tlca.dynamic_inventory_df["flow"] == uptake_flow.id]
    .groupby("date")["amount"]
    .sum()
    .sort_index()
)

fig, ax = plt.subplots(figsize=(13, 3))
ax.plot(uptake_over_time.index, uptake_over_time.values, marker="o", linestyle="none")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("kg CO2 taken up")
plt.tight_layout()
plt.show()

## 2 | Radiative forcing over time

`metric="radiative_forcing"` characterizes each emission from the moment it happens, without
integrating anything away. `bw_timex` maps the IPCC AR6 functions to the biosphere flows of
`METHOD` by itself, so there is no `characterization_functions` dict to pass here - we are on
ecoinvent flows.


In [ ]:
# instantaneous, one series per emitting activity
tlca.dynamic_lcia(metric="radiative_forcing", time_horizon=100)
tlca.plot_dynamic_characterized_inventory(sum_emissions_within_activity=True)

The same characterization, now summed over all activities and **accumulated** - the
warming the house has caused up to each point in time, rather than in each single year:


In [ ]:
tlca.plot_dynamic_characterized_inventory(sum_activities=True, cumsum=True)

The house life cycle spends its first decades **cooling**: the tree's uptake is characterized
before anything is emitted. The curve crosses zero only years after the house's construction.


## 3 | 🛠️ Your turn: fixed or flexible time horizon?

Both options say "100 years", and they mean different things:

| | window for one emission |
|---|---|
| `fixed_time_horizon=False` (default) | 100 years starting **at that emission** |
| `fixed_time_horizon=True` (Levasseur) | up to the **functional unit's date + 100 years**, whoever emits |

This system is the interesting case, because its flows are spread over more than a century:
the tree takes up CO2 up to 40 years *before* the functional unit, the incineration happens
50 years *after* it.

**Predict first, then run:** does `fixed_time_horizon=True` give a higher or a lower GWP100
than the default - and why?


In [ ]:
for fixed in (False, True):
    tlca.dynamic_lcia(metric="GWP", time_horizon=100, fixed_time_horizon=fixed)
    print(f"fixed_time_horizon={fixed!s:<5}  GWP100: {tlca.dynamic_score:,.0f} kg CO2-eq")

**Lower**, and by a lot: about 13,700 instead of 24,700 kg CO2-eq.

The uptake is what moves. With the fixed horizon, the CO2 the tree took up 40 years before
move-in is counted towards a window that ends in 2125, so it is credited for ~140 years
instead of 100 - the cooling contribution grows. The incineration in 2075, on the other hand,
gets only its remaining ~50 years instead of a full century, so its warming shrinks. Both
effects push the score down.

Neither number is "the right one". `False` is what conventional LCIA does and what published
GWP factors mean; `True` is the consistent choice when you want one common cut-off date for
the whole study - and the one to use when you compare systems whose emissions sit at
different points in time.


The same two variants as **radiative forcing over time**, in two panels because the
two quantities differ by a factor of ~70 and would hide each other on one axis:

- **top**: the radiative forcing *in* each year - the instantaneous warming effect of
  everything emitted so far, as it decays.
- **bottom**: the running sum of the top panel. This is the area a GWP integrates into its
  single number, which is why the two curves end where the two GWP100 scores were.

The dotted lines are move-in (2025) and the common cut-off the fixed horizon uses
(2125 = functional unit + 100 years):


In [ ]:
import pandas as pd

series = {}
for fixed in (False, True):
    tlca.dynamic_lcia(metric="radiative_forcing", time_horizon=100, fixed_time_horizon=fixed)
    rf = tlca.characterized_inventory.groupby("date")["amount"].sum().sort_index()
    series[fixed] = rf.resample("YS").sum()  # monthly resolution is noise at this scale

# flexible drawn solid and underneath, fixed dashed on top: wherever the two agree, the
# dashes sit straight on the blue line
STYLES = {
    False: dict(color="tab:blue", linewidth=2.2, linestyle="-",
                label="flexible: 100 years from each emission"),
    True: dict(color="tab:orange", linewidth=1.6, linestyle="--",
               label="fixed: everything counted until 2125"),
}

fig, (ax_year, ax_cum) = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

for fixed, style in STYLES.items():
    rf = series[fixed]
    ax_year.plot(rf.index, rf.values, **style)
    ax_cum.plot(rf.index, rf.cumsum().values, **style)

ax_year.set_title("instantaneous - radiative forcing in each year")
ax_cum.set_title("cumulative - the running sum, i.e. what a GWP integrates")

for ax in (ax_year, ax_cum):
    ax.axhline(0, color="black", linewidth=0.8)
    bottom = ax.get_ylim()[0]
    for year, label in [(2025, "move-in"), (2125, "FU + 100 a")]:
        date = pd.Timestamp(f"{year}-01-01")
        ax.axvline(date, color="grey", linestyle=":", linewidth=1)
        ax.text(date, bottom, f" {label}", color="grey", fontsize=9, va="bottom")
    ax.set_ylabel("W/m2")
    ax.legend(loc="upper left", framealpha=0.9)

plt.tight_layout()
plt.show()

For most of the century the dashes sit straight on the solid line: every flow is
still inside both windows, so both runs characterize it identically. They come apart around
2100 in two steps.

First the dashed line drops **below** the solid one. That is the tree: its uptake is negative
forcing, the flexible run closes the 100-year window on it around 2085, and the fixed run
keeps counting it to 2125 - about 140 years of credit instead of 100. Then, at 2125, the
fixed run stops altogether, while the flexible one carries the 2075 incineration on to 2174.

Cooling counted longer, warming counted shorter: that is the whole of the 24,711 vs 13,704
gap. This value corresponds to the integral of the curve in the bottom panel.



### 🛠️ Your turn: how much does the horizon length itself matter?

Use [`TimexLCA.compare()`](https://docs.brightway.dev/projects/bw-timex/en/latest/content/api/bw_timex/timex_lca/index.html)
to characterize the sommerhus inventory with GWP over **20, 50, 100 and 500 years, each with `fixed_time_horizon` False and True**.

**Predict first, then run:** which of the eight numbers comes out **negative**, and what
happens to the gap between the two columns as the horizon grows?


In [ ]:
from dataclasses import replace

from bw_timex import TimexLCASettings

base = TimexLCASettings(
    demand={living: 1},
    method=METHOD,
    timeline={"starting_datetime": "2025-01-01", "temporal_grouping": "month"},
    # the static score does not depend on the horizon, so don't pay for it eight times
    lcia={"metric": "GWP", "static_lcia_enabled": False},
)

comparison = TimexLCA.compare(
    [
        replace(
            base,
            time_horizon=horizon,
            fixed_time_horizon=fixed,
            label=f"GWP{horizon}, {'fixed' if fixed else 'flexible'}",
        )
        for horizon in (20, 50, 100, 500)
        for fixed in (False, True)
    ]
)

comparison.summary.pivot(
    index="time_horizon", columns="fixed_time_horizon", values="dynamic_score"
)

Read the `fixed_time_horizon=True` column downwards and watch the warning `bw_timex`
printed: with a fixed horizon of 20 years, everything after 2045 lies **outside** the window
and is dropped entirely - the grid draw of the 2050s, the incineration, all of it. What is
left is mostly the tree, so the score turns negative. It is not a claim that the house is
climate positive; it is the honest answer to the question "what happens between 2025 and
2045", and a good reason to state your horizon and its start whenever you report a dynamic
score.

The two columns converge towards long horizons: a 500-year window makes a 40-year offset in
the timing almost irrelevant.


## 4 | 🛠️ Your turn: a characterization function of your own

Nothing in `characterize()` is climate-specific - it applies whatever function you map onto a
flow. So let us give the *sommerhus* something that climate metrics cannot see.

A Danish summer house draws its water from **its own well**, and it is lived in during the
**summer** - exactly when the groundwater is under most pressure. Static LCIA has one factor per flow and cannot express that, but a dynamic characterization
function can.

**(a) Add a water flow to the model.** 60 m3 of `"Water, well, in ground"` per year of
habitation, drawn **April to September** - the house fills up and the garden needs watering -
with the peak in **July and August**. Say:

| Apr | May | Jun | Jul | Aug | Sep |
|---|---|---|---|---|---|
| 7% | 12% | 20% | 27% | 24% | 10% |

Then rebuild the `TimexLCA` so the inventory contains it.

> The flow lives in `bd.config.biosphere`, categories `("natural resource", "in water")`.
> A biosphere edge is `living.new_edge(input=..., amount=..., type="biosphere")`, and its
> `temporal_distribution` works exactly like the ones on technosphere edges: months relative
> to the consumer, shares that sum to 1 **over the whole exchange**. 


In [ ]:
import numpy as np
from bw_timex import TemporalDistribution

WATER_PER_YEAR = 60  # m3/a, drawn from the house's own well

# looked up again, so this cell also works if `foreground` was rebuilt in between
living = bd.get_node(database="foreground", name="living in the sommerhus")
water_flow = bd.get_node(
    database=bd.config.biosphere,
    name="Water, well, in ground",
    categories=("natural resource", "in water"),
)

# drop a previous attempt, so running this cell twice does not draw the water twice
for edge in list(living.biosphere()):
    if edge.input.id == water_flow.id:
        edge.delete()

# the season, as months after a January move-in: April ... September
SEASON_MONTHS = np.array([3, 4, 5, 6, 7, 8])
SEASON_SHARES = np.array([0.07, 0.12, 0.20, 0.27, 0.24, 0.10])  # peak in July/August

# ... repeated for every year of habitation, each year taking 1/50 of the total
year_offsets = np.repeat(np.arange(LIFETIME) * 12, len(SEASON_MONTHS))
td_water = TemporalDistribution(
    date=(np.tile(SEASON_MONTHS, LIFETIME) + year_offsets).astype("timedelta64[M]"),
    amount=np.tile(SEASON_SHARES, LIFETIME) / LIFETIME,
)

water_edge = living.new_edge(
    input=water_flow, amount=WATER_PER_YEAR * LIFETIME, type="biosphere"
)
water_edge["temporal_distribution"] = td_water
water_edge.save()
bd.Database("foreground").process()

tlca = TimexLCA({living: 1}, METHOD)
tlca.build_timeline(starting_datetime="2025-01-01", temporal_grouping="month")
tlca.lci()

**(b) Characterize it.** You already wrote this function: `characterize_water_scarcity`
from [`4_dynamic_characterization.ipynb`](../../4_dynamic_characterization.ipynb), with
`water_stress_index_by_month`. Copy both across - nothing about them changes here. The only new part is what you point it at.

> Call `characterize()` on `tlca.dynamic_inventory_df` directly, with
> `characterization_functions={water_flow.id: characterize_water_scarcity}`.


In [ ]:
from dynamic_characterization import characterize
from dynamic_characterization.classes import CharacterizedRow

# straight from the dynamic-characterization notebook, unchanged
water_stress_index_by_month = [0.2, 0.3, 0.4, 0.5, 0.7, 0.9, 1.0, 1.0, 0.8, 0.6, 0.4, 0.3]


def characterize_water_scarcity(series, period: int = 1) -> CharacterizedRow:
    """A toy seasonal water scarcity characterization function"""
    month = series.date.month
    weight = water_stress_index_by_month[month - 1]
    impact = series.amount * weight

    return CharacterizedRow(
        date=np.array([series.date.to_datetime64()], dtype="datetime64[s]"),
        amount=np.array([impact], dtype="float64"),
        flow=series.flow,
        activity=series.activity,
    )


# ... and now pointed at a real supply chain instead of three dummy rows
water_rows = tlca.dynamic_inventory_df.query("flow == @water_flow.id")
water_scarcity = characterize(
    tlca.dynamic_inventory_df,
    characterization_functions={water_flow.id: characterize_water_scarcity},
)

print(f"withdrawn:  {water_rows['amount'].sum():,.0f} m3 over {len(water_rows)} withdrawals")
print(f"weighted:   {water_scarcity['amount'].sum():,.0f} stress-m3")

The season sits on the high half of the index: weighted by the withdrawal curve it
averages 0.89, against an annual mean of 0.61. So the seasonal indicator comes out about
**1.5x** the season-blind one (2,679 against 1,786 stress-m3) - a factor built on the annual
average would report this house a third lower. Nothing about the inventory changed, only the
question the characterization function asks of it.

Two details worth noticing:

- the water is slightly *more* than your 3,000 m3. The same flow occurs in the **background**
  too: glass wool, heat pumps and electricity all draw well water, and those withdrawals sit
  in the inventory with their own dates. Your own edge is the dominant part, not all of it.
- the month only survives because the characterization ran on `dynamic_inventory_df`.
  Through `dynamic_lcia()` every withdrawal would have landed on 1 January and the index
  would always have read 0.2.


## 5 | 🚀 Outlook, for the ambitious: make it prospective

The index you just used is a snapshot of *today's* summer. It will not hold for 2075: drier
summers, a lower water table, more neighbours on the same aquifer. The climate functions have
the same problem, and the Watanabe pCFs from the previous notebook solve it by letting the
characterization factor depend on the **year** of the emission as well as the substance.

Do the same for the water. Write `characterize_water_scarcity_prospective(series, period=1)`
that keeps the month lookup and multiplies it by a trend read off the year - say stress rises
by 60% between 2025 and 2075 and is held flat outside that range - then characterize the same
inventory with it and compare.

> `np.interp(year, [2025, 2075], [1.0, 1.6])` gives you the trend and clamps outside the
> range, exactly like the temporal evolution factors in notebook 3. The month must survive:
> `series.date` still carries it, so keep reading `series.date.month` and do **not** route
> this through `dynamic_lcia()`.
>
> Worth asking yourself afterwards: this house draws the same 60 m3 every summer for 50 years.
> Under a rising index, is its *late* water worth more than its early water - and what would
> that mean for a house built in 2045 instead?


In [ ]:
def characterize_water_scarcity_prospective(series, period: int = 1) -> CharacterizedRow:
    """Seasonal weight, times a scarcity trend read off the year of the withdrawal."""
    month = series.date.month
    weight = water_stress_index_by_month[month - 1]
    trend = np.interp(series.date.year, [2025, 2075], [1.0, 1.6])  # flat outside the range

    return CharacterizedRow(
        date=np.array([series.date.to_datetime64()], dtype="datetime64[s]"),
        amount=np.array([series.amount * weight * trend], dtype="float64"),
        flow=series.flow,
        activity=series.activity,
    )


water_scarcity_prospective = characterize(
    tlca.dynamic_inventory_df,
    characterization_functions={water_flow.id: characterize_water_scarcity_prospective},
)

print(f"today's index:     {water_scarcity['amount'].sum():,.0f} stress-m3")
print(f"rising index:      {water_scarcity_prospective['amount'].sum():,.0f} stress-m3")

In [ ]:
if "water_scarcity_prospective" in globals():
    per_year = {
        "today's index": water_scarcity.assign(year=water_scarcity["date"].dt.year)
        .groupby("year")["amount"].sum(),
        "rising index": water_scarcity_prospective.assign(
            year=water_scarcity_prospective["date"].dt.year
        ).groupby("year")["amount"].sum(),
    }

    fig, ax = plt.subplots(figsize=(11, 3.5))
    for label, series_ in per_year.items():
        ax.plot(series_.index, series_.values, marker="o", markersize=3, label=label)
    ax.set_xlabel("year of withdrawal")
    ax.set_ylabel("stress-weighted m3 per year")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("nothing to compare yet - section 5 is still open")

Same water, same months, same inventory: only the year the withdrawal happens in now
matters. The early summers are unchanged and the later ones weigh up to 60% more, which adds
up to **2,679 -> 3,465 stress-m3**, +29% over the lifetime. The house's water burden is
**back-loaded** - the water version of exactly the point the dynamic climate metrics make, and
a house built in 2045 would draw all of its water on the expensive part of that curve.


## 6 | Where to go from here

- **Prospective characterization factors** (Watanabe et al. 2026): same call, `metric="pGWP"`
  or `"prospective_radiative_forcing"` after `prospective.set_scenario(...)`. On a system whose
  emissions run to 2125, the scenario-dependent radiative efficiencies are not a detail.
- **`fixed_time_horizon` in a comparison**: two houses built decades apart are exactly the case
  where the choice changes the ranking, not just the number.
- **Your own metric**: the water function above is 8 lines. Anything that depends on *when* -
  seasonal water, noise at night, harvest timing - fits the same shape.
